<a href="https://colab.research.google.com/github/MoharanaSudhanshu/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MoharanaSudhanshu/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding 1 — AI-driven search visibility is changing

One finding from the FlyRank research paper that I want to examine is the paper's discussion of how AI-driven search is changing the way users discover information and how brands need to think beyond traditional search.

### Methodology question

I would want to understand exactly how this finding was measured. In particular, I would want to know what data sources and sample were used, how "AI-driven search" was defined, and over what time period the measurements were collected. I would also want to know whether the validation or comparison design supports extending the observed result to the broader search market.

## Finding 2 — AI search creates a different measurement challenge

A second finding I want to examine is the paper's discussion of measuring visibility and performance in AI-driven search environments.

### Methodology question

I would want to know how the underlying visibility or performance measure was defined and whether the measurement method captures the different ways users interact with AI search systems. I would also want to understand the sampling process and whether the reported result is descriptive of the observed dataset or intended as a broader generalization.

### Why I am asking these questions

These are not claims that the research is incorrect. They are questions I would ask before applying the findings to my own model. I want to distinguish between what was directly observed in the research data and what can reasonably be generalized beyond that data.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

!git clone https://github.com/MoharanaSudhanshu/flyrank-ml-internship.git



fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [6]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# ==========================================================
# 1. LOAD DATA
# ==========================================================

REPO_PATH = "/content/flyrank-ml-internship"

DATA_PATH = os.path.join(
    REPO_PATH,
    "data",
    "raw",
    "content_refresh_anonymized.csv"
)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found:\n{DATA_PATH}"
    )

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully")
print("Shape:", df.shape)


# ==========================================================
# 2. CREATE THE SAME TARGET AS WEEK 5
# ==========================================================

df["refresh_needed"] = (
    (df["ctr"] < 2.0) &
    (df["avg_position"] > 10) &
    (df["trend_direction"] == "down")
).astype(int)

print("\nTarget distribution:")
print(df["refresh_needed"].value_counts())


# ==========================================================
# 3. SAME FEATURES AS WEEK 5
# ==========================================================

categorical = [
    "competition_level",
    "content_type",
    "main_intent",
    "provider_used",
    "model_used",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier",
    "trend_direction"
]

df_ml = df.copy()

for col in categorical:
    encoder = LabelEncoder()
    df_ml[col] = encoder.fit_transform(
        df_ml[col].astype(str)
    )


features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "engagement_rate",
    "ctr",
    "avg_position",
    "scroll_rate",
    "trend_pct",
    "content_age_days",
    "days_since_last_update"
]

X = df_ml[features]
y = df_ml["refresh_needed"]

groups = df["client_id"]

print("\nFeature matrix:", X.shape)
print("Target:", y.shape)
print("Unique clients:", groups.nunique())


# ==========================================================
# 4. WEEK-5 STYLE RANDOM 80/20 SPLIT
# ==========================================================

X_train_random, X_test_random, y_train_random, y_test_random = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)


# ==========================================================
# 5. TRAIN RANDOM FOREST ON RANDOM SPLIT
# ==========================================================

random_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

random_model.fit(
    X_train_random,
    y_train_random
)

random_predictions = random_model.predict(
    X_test_random
)


# ==========================================================
# 6. RANDOM SPLIT METRICS
# ==========================================================

random_accuracy = accuracy_score(
    y_test_random,
    random_predictions
)

random_precision = precision_score(
    y_test_random,
    random_predictions,
    zero_division=0
)

random_recall = recall_score(
    y_test_random,
    random_predictions,
    zero_division=0
)

random_f1 = f1_score(
    y_test_random,
    random_predictions,
    zero_division=0
)

print("\n========================================")
print("WEEK-5 STYLE RANDOM SPLIT")
print("========================================")

print("Accuracy :", round(random_accuracy, 4))
print("Precision:", round(random_precision, 4))
print("Recall   :", round(random_recall, 4))
print("F1 Score :", round(random_f1, 4))


# ==========================================================
# 7. HONEST GROUPED SPLIT BY CLIENT
# ==========================================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        X,
        y,
        groups=groups
    )
)

X_train_grouped = X.iloc[train_idx]
X_test_grouped = X.iloc[test_idx]

y_train_grouped = y.iloc[train_idx]
y_test_grouped = y.iloc[test_idx]

train_clients = groups.iloc[train_idx]
test_clients = groups.iloc[test_idx]

print("\n========================================")
print("GROUPED SPLIT CHECK")
print("========================================")

print("Training rows:", len(train_idx))
print("Testing rows :", len(test_idx))

print(
    "Training clients:",
    train_clients.nunique()
)

print(
    "Testing clients:",
    test_clients.nunique()
)

overlap = set(train_clients).intersection(
    set(test_clients)
)

print(
    "Client overlap:",
    len(overlap)
)


# ==========================================================
# 8. TRAIN RANDOM FOREST ON GROUPED SPLIT
# ==========================================================

grouped_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

grouped_model.fit(
    X_train_grouped,
    y_train_grouped
)

grouped_predictions = grouped_model.predict(
    X_test_grouped
)


# ==========================================================
# 9. GROUPED SPLIT METRICS
# ==========================================================

grouped_accuracy = accuracy_score(
    y_test_grouped,
    grouped_predictions
)

grouped_precision = precision_score(
    y_test_grouped,
    grouped_predictions,
    zero_division=0
)

grouped_recall = recall_score(
    y_test_grouped,
    grouped_predictions,
    zero_division=0
)

grouped_f1 = f1_score(
    y_test_grouped,
    grouped_predictions,
    zero_division=0
)

print("\n========================================")
print("HONEST GROUPED SPLIT")
print("========================================")

print("Accuracy :", round(grouped_accuracy, 4))
print("Precision:", round(grouped_precision, 4))
print("Recall   :", round(grouped_recall, 4))
print("F1 Score :", round(grouped_f1, 4))


# ==========================================================
# 10. BEFORE / AFTER COMPARISON
# ==========================================================

comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ],
    "Random 80/20 Split": [
        random_accuracy,
        random_precision,
        random_recall,
        random_f1
    ],
    "Client-Grouped Split": [
        grouped_accuracy,
        grouped_precision,
        grouped_recall,
        grouped_f1
    ]
})

comparison["Random 80/20 Split"] = (
    comparison["Random 80/20 Split"].round(4)
)

comparison["Client-Grouped Split"] = (
    comparison["Client-Grouped Split"].round(4)
)

print("\n========================================")
print("BEFORE / AFTER COMPARISON")
print("========================================")

display(comparison)


# ==========================================================
# 11. GROUPED CLASSIFICATION REPORT
# ==========================================================

print("\n========================================")
print("GROUPED CLASSIFICATION REPORT")
print("========================================")

print(
    classification_report(
        y_test_grouped,
        grouped_predictions,
        zero_division=0
    )
)

Dataset loaded successfully
Shape: (30000, 44)

Target distribution:
refresh_needed
0    21116
1     8884
Name: count, dtype: int64

Feature matrix: (30000, 15)
Target: (30000,)
Unique clients: 32

WEEK-5 STYLE RANDOM SPLIT
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0

GROUPED SPLIT CHECK
Training rows: 23837
Testing rows : 6163
Training clients: 25
Testing clients: 7
Client overlap: 0

HONEST GROUPED SPLIT
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0

BEFORE / AFTER COMPARISON


,Metric,Random 80/20 Split,Client-Grouped Split
0,Accuracy,1.0,1.0
1,Precision,1.0,1.0
2,Recall,1.0,1.0
3,F1 Score,1.0,1.0



GROUPED CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4729
           1       1.00      1.00      1.00      1434

    accuracy                           1.00      6163
   macro avg       1.00      1.00      1.00      6163
weighted avg       1.00      1.00      1.00      6163



## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## Leakage Audit

The Week-5 target was constructed using CTR, average position, and trend direction. These same variables were also included as model features. This creates a direct leakage risk because the model receives information that was used to define the label.

The perfect performance observed under both the random and client-grouped splits is therefore not interpreted as evidence of a genuinely perfect predictive model.

I performed a leakage audit by identifying the target-defining features and removing them from the model. I also excluded position tier and trend percentage as possible proxy variables related to the target definition.

The resulting leakage-safe model was evaluated using the same client-grouped split. The comparison between the original and leakage-safe models shows how much of the original performance depended on information directly related to the target construction.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ==========================================================
# SECTION 3 — LEAKAGE AUDIT
# ==========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ==========================================================
# 1. TARGET-DEFINING FEATURES
# ==========================================================

target_defining_features = [
    "ctr",
    "avg_position",
    "trend_direction"
]

print("Target-defining features:")
for feature in target_defining_features:
    print("-", feature)


# ==========================================================
# 2. CHECK WHETHER TARGET FEATURES ARE IN MODEL FEATURES
# ==========================================================

print("\n========================================")
print("DIRECT LEAKAGE CHECK")
print("========================================")

for feature in target_defining_features:

    if feature in features:
        print(
            f"LEAKAGE RISK: {feature} is used "
            f"to create the target AND as a model feature."
        )
    else:
        print(
            f"OK: {feature} is not used as a model feature."
        )


# ==========================================================
# 3. IDENTIFY DERIVED / PROXY FEATURES
# ==========================================================

possible_proxy_features = [
    "position_tier",
    "trend_pct",
    "impression_tier",
    "freshness_tier"
]

print("\n========================================")
print("POSSIBLE PROXY FEATURES")
print("========================================")

for feature in possible_proxy_features:

    if feature in features:
        print(
            f"REVIEW: {feature} may contain information "
            f"related to the target definition."
        )
    else:
        print(
            f"NOT USED: {feature}"
        )


# ==========================================================
# 4. CREATE LEAKAGE-SAFE FEATURE SET
# ==========================================================

leakage_features = [
    "ctr",
    "avg_position",
    "trend_direction",
    "position_tier",
    "trend_pct"
]

safe_features = [
    feature
    for feature in features
    if feature not in leakage_features
]

print("\n========================================")
print("LEAKAGE-SAFE FEATURES")
print("========================================")

print("Original feature count:", len(features))
print("Safe feature count:", len(safe_features))

print("\nRemoved features:")
for feature in features:
    if feature not in safe_features:
        print("-", feature)

print("\nRemaining features:")
for feature in safe_features:
    print("-", feature)


# ==========================================================
# 5. PREPARE LEAKAGE-SAFE DATA
# ==========================================================

X_safe = df_ml[safe_features].copy()

y_safe = df_ml["refresh_needed"].copy()

groups_safe = df["client_id"].copy()


# ==========================================================
# 6. HANDLE MISSING VALUES
# ==========================================================

X_safe = X_safe.replace(
    [np.inf, -np.inf],
    np.nan
)

X_safe = X_safe.fillna(
    X_safe.median(numeric_only=True)
)

X_safe = X_safe.fillna(0)


# ==========================================================
# 7. GROUPED TRAIN / TEST SPLIT
# ==========================================================

gss_safe = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx_safe, test_idx_safe = next(
    gss_safe.split(
        X_safe,
        y_safe,
        groups=groups_safe
    )
)

X_train_safe = X_safe.iloc[train_idx_safe]
X_test_safe = X_safe.iloc[test_idx_safe]

y_train_safe = y_safe.iloc[train_idx_safe]
y_test_safe = y_safe.iloc[test_idx_safe]


# ==========================================================
# 8. TRAIN LEAKAGE-SAFE RANDOM FOREST
# ==========================================================

safe_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

safe_model.fit(
    X_train_safe,
    y_train_safe
)

safe_predictions = safe_model.predict(
    X_test_safe
)


# ==========================================================
# 9. EVALUATE LEAKAGE-SAFE MODEL
# ==========================================================

safe_accuracy = accuracy_score(
    y_test_safe,
    safe_predictions
)

safe_precision = precision_score(
    y_test_safe,
    safe_predictions,
    zero_division=0
)

safe_recall = recall_score(
    y_test_safe,
    safe_predictions,
    zero_division=0
)

safe_f1 = f1_score(
    y_test_safe,
    safe_predictions,
    zero_division=0
)


print("\n========================================")
print("LEAKAGE-SAFE MODEL PERFORMANCE")
print("========================================")

print(
    "Accuracy :",
    round(safe_accuracy, 4)
)

print(
    "Precision:",
    round(safe_precision, 4)
)

print(
    "Recall   :",
    round(safe_recall, 4)
)

print(
    "F1 Score :",
    round(safe_f1, 4)
)


# ==========================================================
# 10. COMPARE ORIGINAL VS LEAKAGE-SAFE MODEL
# ==========================================================

leakage_comparison = pd.DataFrame({

    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ],

    "Original Week-5 Model": [
        grouped_accuracy,
        grouped_precision,
        grouped_recall,
        grouped_f1
    ],

    "Leakage-Safe Model": [
        safe_accuracy,
        safe_precision,
        safe_recall,
        safe_f1
    ]
})

leakage_comparison[
    "Original Week-5 Model"
] = leakage_comparison[
    "Original Week-5 Model"
].round(4)

leakage_comparison[
    "Leakage-Safe Model"
] = leakage_comparison[
    "Leakage-Safe Model"
].round(4)


print("\n========================================")
print("LEAKAGE AUDIT COMPARISON")
print("========================================")

display(leakage_comparison)

Target-defining features:
- ctr
- avg_position
- trend_direction

DIRECT LEAKAGE CHECK
LEAKAGE RISK: ctr is used to create the target AND as a model feature.
LEAKAGE RISK: avg_position is used to create the target AND as a model feature.
OK: trend_direction is not used as a model feature.

POSSIBLE PROXY FEATURES
NOT USED: position_tier
REVIEW: trend_pct may contain information related to the target definition.
NOT USED: impression_tier
NOT USED: freshness_tier

LEAKAGE-SAFE FEATURES
Original feature count: 15
Safe feature count: 12

Removed features:
- ctr
- avg_position
- trend_pct

Remaining features:
- search_volume
- competition
- cpc
- word_count
- char_count
- impressions_90d
- clicks_90d
- sessions_90d
- engagement_rate
- scroll_rate
- content_age_days
- days_since_last_update

LEAKAGE-SAFE MODEL PERFORMANCE
Accuracy : 0.7334
Precision: 0.3083
Recall   : 0.1172
F1 Score : 0.1698

LEAKAGE AUDIT COMPARISON


,Metric,Original Week-5 Model,Leakage-Safe Model
0,Accuracy,1.0,0.7334
1,Precision,1.0,0.3083
2,Recall,1.0,0.1172
3,F1 Score,1.0,0.1698


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Claim Rewrite

My original claim was that the Random Forest could reliably predict which pages need a refresh.

A safer claim is:

**The model achieved perfect performance on the evaluated split, but the result is not evidence of perfect real-world prediction because the target was constructed using variables that were also used as model features. After the leakage audit, the leakage-safe evaluation provides a more appropriate estimate of model performance.**

This result is best treated as decision-support evidence rather than proof of reliable prediction.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.